# Additional Material
**A second force field: GROMOS 54A7**


Authors:

- Prof. Marco A. Deriu (marco.deriu@polito.it)
- Eric A. Zizzi (eric.zizzi@polito.it)
- Marcello Miceli (marcello.miceli@polito.it)

# Table of Contents

1. Why a second force field
2. Building the system with GROMOS 54A7
3. Minimisation and equilibration
4. Production
5. AMBER and GROMOS side by side
6. Your report

**Learning outcomes:**
- explain what a force field fixes, and why its simulation settings belong to it
- set up the same molecule with a united-atom force field and its water model
- compare two force fields on the same observables, and say what one run of each can and cannot tell

# Before you start

This notebook is optional. It repeats LAB 3 with a second force field and compares the result with your LAB 3 run, using the observables of LAB 4. You need:

- **LAB 3**: the 300 K run of NMR model 1 with amber99sb-ildn, your own (`../../03-ClassicalMD/cln025_model1`) or the course's (`course-fetch --lab 03-ClassicalMD --solutions`);
- **LAB 4**, sections 1 to 7: the course's fixed definitions of the state of the hairpin, in `../../common/scripts/cln025.py`.

Everything happens in this folder, `~/molbiomech/labs/Additional_Material/gromos54a7`. The production run takes as long as the one of LAB 3.

# 1. Why a second force field

Everything a simulation predicts depends on the force field. For CLN025 the difference is large: in a comparison of four force fields, AMBER ff14SB (a close relative of the course's amber99sb-ildn) stabilised the folded hairpin too much, while GROMOS 54A7 agreed well with the experimental populations of folded and unfolded states and reproduced the experimental folding times best (Kamenik et al., *J Chem Phys* 2020, [doi:10.1063/5.0022135](https://doi.org/10.1063/5.0022135)).

Both force fields come with GROMACS:

| | LAB 3 | This notebook |
|---|---|---|
| Force field | amber99sb-ildn | gromos54a7 |
| Hydrogens | Every hydrogen is an atom | **United atom**: the hydrogens of CH, CH₂ and CH₃ groups are merged into their carbon |
| Water | TIP3P | SPC |
| Electrostatics | PME, 1.0 nm cut-off | Reaction field beyond a 1.4 nm cut-off, with the relative permittivity of the surrounding medium set to 61 |
| Lennard-Jones | 1.0 nm cut-off, dispersion correction | 1.4 nm cut-off, no correction |
| Parameter files | `../../common/mdp/*.mdp` | `../../common/mdp/gromos54a7/*.mdp` |

**A force field comes with the simulation settings it was parametrised with.** GROMOS was parametrised with a reaction field and a 1.4 nm cut-off (Diem & Oostenbrink, *J Chem Theory Comput* 2020, [doi:10.1021/acs.jctc.0c00509](https://doi.org/10.1021/acs.jctc.0c00509)), and a single 1.4 nm cut-off as used by current GROMACS reproduces its behaviour (Silva et al., *J Chem Theory Comput* 2018, [doi:10.1021/acs.jctc.8b00758](https://doi.org/10.1021/acs.jctc.8b00758)). Running GROMOS parameters with AMBER's settings would simulate a different model.

Note that the comparison changes the force field and the water model together: each force field was parametrised with its own water. GROMOS was also parametrised with every bond length constrained; here only the bonds to hydrogen are constrained, as in LAB 3, so that the time step and the rest of the protocol stay the same.

In [ ]:
!diff ../../common/mdp/md.mdp ../../common/mdp/gromos54a7/md.mdp

# 2. Building the system with GROMOS 54A7

## 2.1 Topology

The folder is built as in LAB 3. `build_caps.py --gromos` names the acetyl carbon as GROMOS expects:

In [ ]:
%%bash
mkdir -p cln025_model1_gromos/{00-build,01-em,02-nvt,03-npt,04-md}
cp ../../common/mdp/gromos54a7/*.mdp cln025_model1_gromos/
python3 ../../common/scripts/build_caps.py ../../common/structures/2RVD.pdb 1 cln025_model1_gromos/00-build/cln025_capped.pdb --gromos > /dev/null
cd cln025_model1_gromos/00-build
printf "2\n2\n" | gmx pdb2gmx -f cln025_capped.pdb -o cln025.gro -p topol.top -i posre.itp \
                               -ff gromos54a7 -water spc -ignh -ter > pdb2gmx.out 2>&1 || tail -n 20 pdb2gmx.out
grep -E -A 4 "^Select (start|end) terminus" pdb2gmx.out
grep -E "Now there are|Total charge" pdb2gmx.out

GROMOS attaches the chain ends to the first and last residue, and would treat ACE and NH2 as charged ends. `-ter` makes `pdb2gmx` ask for each end instead, and `printf "2\n2\n"` answers **None** twice: the caps already are the ends of the chain.

<div class="alert alert-block alert-info">The same peptide, the same charge, but how many atoms? Compare with the AMBER topology of LAB 3, and explain the difference.</div>

## 2.2 Box, water and ions

The same commands as in LAB 3, with SPC water. Look at what `grompp` says this time:

In [ ]:
%%bash
cd cln025_model1_gromos/00-build
gmx editconf -f cln025.gro -o box.gro -c -d 1.0 -bt dodecahedron > editconf.out 2>&1 || tail -n 20 editconf.out
gmx solvate -cp box.gro -cs spc216.gro -o solvated.gro -p topol.top > solvate.out 2>&1 || tail -n 20 solvate.out
gmx grompp -f ../em.mdp -c solvated.gro -p topol.top -o ions.tpr > grompp_ions.out 2>&1 || grep -E -A 9 "^(WARNING|NOTE)" grompp_ions.out

Two messages. The net charge is now only a NOTE: without Ewald electrostatics it causes no artefacts, and the ions will remove it anyway. The WARNING is new: GROMACS warns about **every** GROMOS system that the force field was parametrised with a twin-range cut-off scheme that current GROMACS no longer supports. The authors of GROMOS tested exactly this and found the effect minor (Diem & Oostenbrink 2020), and Silva et al. (2018) found the single cut-off consistent with the force field. Having read it and checked the literature, we accept this one warning with `-maxwarn 1` in every GROMOS `grompp`.

In [ ]:
%%bash
cd cln025_model1_gromos/00-build
gmx grompp -f ../em.mdp -c solvated.gro -p topol.top -o ions.tpr -maxwarn 1 > grompp_ions.out 2>&1 || tail -n 20 grompp_ions.out
echo SOL | gmx genion -s ions.tpr -o system.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15 > genion.out 2>&1 || tail -n 20 genion.out
tail -n 4 topol.top
echo "atoms: $(sed -n 2p system.gro)"

# 3. Minimisation and equilibration

The same three stages as in LAB 3, with the GROMOS parameter files:

In [ ]:
%%bash
cd cln025_model1_gromos/01-em
gmx grompp -f ../em.mdp -c ../00-build/system.gro -p ../00-build/topol.top -o em.tpr -maxwarn 1 > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm em -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep -A 3 "converged" em.log

cd ../02-nvt
gmx grompp -f ../nvt.mdp -c ../01-em/em.gro -r ../01-em/em.gro -p ../00-build/topol.top -o nvt.tpr -maxwarn 1 > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm nvt -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep -B 1 "Performance:" nvt.log

cd ../03-npt
gmx grompp -f ../npt.mdp -c ../02-nvt/nvt.gro -r ../02-nvt/nvt.gro -t ../02-nvt/nvt.cpt -p ../00-build/topol.top -o npt.tpr -maxwarn 1 > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm npt -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep -B 1 "Performance:" npt.log
printf "Temperature\nDensity\n" | gmx energy -f npt.edr > energy.out 2>&1 || tail -n 20 energy.out
grep -A 3 "^Energy" energy.out

<div class="alert alert-block alert-info">Is the GROMOS system faster or slower than the AMBER one of LAB 3? It has fewer atoms, but a longer cut-off, so each atom interacts with more neighbours. SPC water is also a little less dense than TIP3P: compare the densities.</div>

# 4. Production

The production run at 300 K, with the same length as your LAB 3 run:

In [ ]:
import re

length_ns = 10      # the length of your LAB 3 run, in ns

nsteps = round(length_ns * 1000 / 0.002)
mdp = open("cln025_model1_gromos/md.mdp").read()
mdp = re.sub(r"(?m)^nsteps .*", f"nsteps          = {nsteps}", mdp)
with open("cln025_model1_gromos/04-md/md_300K.mdp", "w") as f:
    f.write(mdp)
print(f"md_300K.mdp: {nsteps} steps = {length_ns} ns at 300 K")

In [ ]:
%%bash
cd cln025_model1_gromos/04-md
gmx grompp -f md_300K.mdp -c ../03-npt/npt.gro -t ../03-npt/npt.cpt -p ../00-build/topol.top -o md_300K.tpr -maxwarn 1 > grompp.out 2>&1 || tail -n 20 grompp.out
ls

Start it from a terminal, in the background, as in LAB 3:

```bash
cd ~/molbiomech/labs/Additional_Material/gromos54a7/cln025_model1_gromos/04-md
nohup gmx mdrun -deffnm md_300K -ntmpi 1 -v > mdrun.out 2>&1 &
```

If it cannot finish, the course's own run is part of the LAB 3 solutions (`course-fetch --lab 03-ClassicalMD --solutions`), in `../../03-ClassicalMD/solutions/Additional_Material/gromos54a7/`.

Choose here which runs section 5 reads. GROMOS: `.` for your own, `../../03-ClassicalMD/solutions/Additional_Material/gromos54a7` for the course's. AMBER: `../../03-ClassicalMD` for your LAB 3 run, `../../03-ClassicalMD/solutions` for the course's.

In [ ]:
%env GROMOS=.
%env AMBER=../../03-ClassicalMD

# 5. AMBER and GROMOS side by side

## 5.1 The peptide alone

As in LAB 4, keep only the peptide, whole and centred, with its own run input file and a first frame for Python.

The two runs must cover the **same length of time** to be worth comparing, and they need not: the course's AMBER run is 500 ns, while this GROMOS one is as long as you made it. The cell finds the end of the shorter of the two and cuts both there.

In [ ]:
%%bash
mkdir -p runs
last () { gmx check -f $1 2>&1 | awk '/Last frame/ {print $NF; exit}'; }   # end time of a trajectory, in ps
a=$(last $AMBER/cln025_model1/04-md/md_300K.xtc)
g=$(last $GROMOS/cln025_model1_gromos/04-md/md_300K.xtc)
end=$(echo "$a $g" | awk '{print ($1 < $2) ? $1 : $2}')
echo "AMBER run $a ps, GROMOS run $g ps -> comparing the first $end ps of each"
for run in amber:$AMBER/cln025_model1 gromos:$GROMOS/cln025_model1_gromos; do
    name=${run%%:*}; src=${run#*:}/04-md/md_300K
    printf "Protein\nProtein\n" | gmx trjconv -s $src.tpr -f $src.xtc -o runs/$name.xtc -pbc mol -center -e $end > runs/$name.out 2>&1 || tail -n 20 runs/$name.out
    printf "Protein\nProtein\n" | gmx trjconv -s $src.tpr -f $src.xtc -o runs/$name.pdb -pbc mol -center -dump 0 >> runs/$name.out 2>&1
    echo "$name: $(grep -c '^ATOM' runs/$name.pdb) peptide atoms, $(grep 'Performance:' $src.log | awk '{print $2}') ns/day"
done

## 5.2 The state of the hairpin

The course's fixed definitions (LAB 4) use only residues 1–10 and match atoms by name, so they work on both force fields although GROMOS has fewer atoms:

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md

sys.path.append("../../common/scripts")
import cln025

names = {"amber": "amber99sb-ildn", "gromos": "gromos54a7"}
colors = {"amber": "tab:orange", "gromos": "tab:green"}
trajs = {ff: md.load(f"runs/{ff}.xtc", top=f"runs/{ff}.pdb") for ff in names}
obs = {ff: cln025.observables(traj) for ff, traj in trajs.items()}

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
for ff, traj in trajs.items():
    time = np.arange(traj.n_frames) * traj.timestep / 1000
    for ax, key in zip(axes, ["e2e", "q", "rmsd"]):
        ax.plot(time, obs[ff][key], lw=0.6, color=colors[ff], label=names[ff])
    print(f"{names[ff]:<15} end-to-end {obs[ff]['e2e'].mean():.2f} nm, Q {obs[ff]['q'].mean():.2f}, "
          f"RMSD from NMR {obs[ff]['rmsd'].mean():.3f} nm, folded fraction {cln025.folded(obs[ff]).mean():.2f}")
axes[0].set_ylabel("end-to-end (nm)")
axes[1].set_ylabel("Q")
axes[1].axhline(cln025.Q_FOLDED, color="black", ls="--", lw=0.8)
axes[2].set_ylabel("RMSD from NMR (nm)")
axes[2].axhline(cln025.RMSD_FOLDED, color="black", ls="--", lw=0.8)
axes[2].set_xlabel("time (ns)")
axes[0].legend()
fig.tight_layout()
fig.savefig("forcefields.png", dpi=300)

## 5.3 The three backbone hydrogen bonds

The hydrogen bonds of LAB 2 and LAB 4, counted as formed when the nitrogen and the oxygen are closer than 0.35 nm. GROMOS keeps the polar hydrogens, so the backbone N and O have the same names in both force fields:

In [ ]:
pairs = {"Asp3 N - Thr8 O": ((3, "N"), (8, "O")), "Gly7 N - Asp3 O": ((7, "N"), (3, "O")), "Thr8 N - Asp3 O": ((8, "N"), (3, "O"))}
print(f"{'fraction of frames formed':<26}" + "".join(f"{p:>17}" for p in pairs))
for ff, traj in trajs.items():
    index = {(a.residue.resSeq, a.name): a.index for a in traj.topology.atoms}
    d = md.compute_distances(traj, [(index[a], index[b]) for a, b in pairs.values()])
    print(f"{names[ff]:<26}" + "".join(f"{f:17.2f}" for f in (d < 0.35).mean(axis=0)))

<div class="alert alert-block alert-info">
Do the two force fields keep the hairpin folded equally well over the window you compared? Which observable differs most? Before concluding, remember what one run can tell: CLN025 folds and unfolds on a time scale of hundreds of nanoseconds or more, so a run of a few nanoseconds that starts folded measures the stability of the fold over those nanoseconds, not the equilibrium population. The course's AMBER run is 500 ns long and still never unfolds (LAB 4 section 8), and Kamenik et al. (2020) needed microseconds of sampling per force field. How many runs like these, or how long, would you need to see the difference they report?
</div>

# 6. Your report

1. The number of atoms of the two systems and the ns/day of the two runs, with an explanation of the differences.
2. The plot of section 5.2 and the table of section 5.3, with two or three sentences: what differs between the force fields, and what one run of each cannot tell you.